# Exercise 7 - Non-linear Classification
## Part 1 - Two Moons

Estimated time: **45-50 minutes**

In the first part of this exercise you will reproduce the Two Moons example from the slides.

The make_moons function comes from [Scikit-learn](http://scikit-learn.org) and is described [here](http://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html).

First run the code to generate the dataset. Note that the dataset is being imported from Scikit-learn. Each input value is a pair of co-ordinates, which you could think of as (x, y) co-ordinates, but note we are using the name x for the network inputs and y for the network outputs. The output values are one-hot encoded, as is usual for classification networks. As you do this exercise, feel free to experiment with the number of samples and the noise.

- Recommended Hardware accelerator for exercise: **CPU or T4 GPU**

In [ ]:
%matplotlib inline
import numpy as np
np.random.seed(0)
from sklearn import datasets
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical

m = 1000

train_x, train_y = datasets.make_moons(n_samples=m, noise=0.15, random_state=0)
colors = ['steelblue' if label == 1 else 'darkred' for label in train_y]
plt.figure(1, figsize=(15, 10))
plt.scatter(train_x[:,0], train_x[:,1], color=colors)
plt.show()

train_x = train_x.astype(np.float32)
train_y = to_categorical(train_y)

print('x.shape =', train_x.shape, 'y.shape =', train_y.shape)

Now run the following network, which performs **linear** classification on the dataset and plots the result.

This cell will take some time to execute. 

In [ ]:
import tensorflow
from tensorflow.keras.models     import Sequential
from tensorflow.keras.layers     import Dense, Input
from tensorflow.keras.optimizers import SGD

tensorflow.keras.backend.clear_session()

n_features = 2
n_hidden = 8
n_labels = 2

model = Sequential()
model.add(Input(shape=(n_features,)))
model.add(Dense( units=n_hidden))
model.add(Dense(units=n_labels, activation='softmax'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer=SGD(learning_rate=0.05), metrics=['accuracy'])

model.fit(train_x, train_y, epochs=1000, batch_size=m, verbose=0)

loss_and_acc = model.evaluate(train_x, train_y, batch_size=m, verbose=0)
print(f'Accuracy = {loss_and_acc[1]:4.2f}')

softmax = model.predict(train_x, batch_size=m)

c = []

for i in range(m):
    if softmax[i,0] > 0.8:
        c.append('darkred')
    elif softmax[i,1] > 0.8:
        c.append('steelblue')
    else:
        c.append('y')

plt.figure(1, figsize=(15, 10))
plt.scatter(train_x[:,0], train_x[:,1], color=c)
plt.show()

Modify the network above to perform non-linear classification. It would be best to start by copy-and-pasting the code above into the box below. All you need to do is insert a ReLU activation function after the hidden layer. Experiment with the number of hidden units and the number of training steps until you get a good result.

In [ ]:
#

#### Solution

Here is our answer. Do not run the cell below unless you want to see the answer we provide!

<details>
    <summary> Click here to view the answer</summary>

    import tensorflow
    from tensorflow.keras.models     import Sequential
    from tensorflow.keras.layers     import Dense, Input
    from tensorflow.keras.optimizers import SGD

    tensorflow.keras.backend.clear_session()

    n_features = 2
    n_hidden = 8
    n_labels = 2

    model = Sequential()
    model.add(Input(shape=(n_features,)))
    model.add(Dense(units=n_hidden, activation='relu'))
    model.add(Dense(units=n_hidden, activation='relu'))
    model.add(Dense(units=n_hidden, activation='relu'))
    model.add(Dense(units=n_labels, activation='softmax'))
    model.summary()

    model.compile(loss='binary_crossentropy', optimizer=SGD(learning_rate=0.1), metrics=['accuracy'])

    model.fit(train_x, train_y, epochs=1000, batch_size=m, verbose=0)

    loss_and_acc = model.evaluate(train_x, train_y, batch_size=m, verbose=0)
    print(f'Accuracy = {loss_and_acc[1]:4.2f}')

    softmax = model.predict(train_x, batch_size=m)

    c = []

    for i in range(m):
        if softmax[i,0] > 0.8:
            c.append('darkred')
        elif softmax[i,1] > 0.8:
            c.append('steelblue')
        else:
            c.append('y')

    plt.figure(1, figsize=(15, 10))
    plt.scatter(train_x[:,0], train_x[:,1], color=c)
    plt.show()

    
</details>


## Part 2 - Clusters of Colored Points

In the second part of this exercise you will start by reproducing the example from the slides in which a set of points in a plane are classified according to their color.

First, run the code to generate and plot the dateset as shown on the slides.

In [ ]:
%matplotlib inline
import random
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import axes3d
from matplotlib import cm

n_features = 2   # The 2 dimensions of each training data point
n_labels   = 3   # The number of categories, shown in various colors (up to 6)
n_clusters = 2   # The number of clusters of each color
spread     = 10  # The maximum distance between the clusters

m          = 102 # The number of datapoints

m = (m // n_labels // n_clusters) * n_labels * n_clusters

rng = np.random.RandomState(seed=6)

x = np.array((rng.standard_normal(m), rng.standard_normal(m)))
x = np.transpose(x)
y = np.empty((m))

batch = m // n_labels // n_clusters

for i in range(n_labels):
    for j in range(n_clusters):
        x[(n_clusters*i+j)*batch:(n_clusters*i+1+j)*batch,0] += rng.randint(-spread,+spread)
        x[(n_clusters*i+j)*batch:(n_clusters*i+1+j)*batch,1] += rng.randint(-spread,+spread)
    y[i*m//n_labels:(i+1)*m//n_labels] = i

#y = (np.arange(n_labels) == y[:,None]).astype(np.float32)
y = to_categorical(y)

indices = np.arange(m)
random.shuffle(indices)

train_x = np.empty((m,n_features)).astype(np.float32)
train_y = np.empty((m,n_labels))

c = [None]*m

for i in range(m):
    train_x[i] = x[indices[i]]
    train_y[i] = y[indices[i]]

    ix = np.argmax(train_y[i,:])   # The color with the highest probability
    c[i] = ('r', 'g', 'b', 'm', 'c', 'y')[ix]

plt.figure(1, figsize=(15, 10))
plt.scatter(train_x[:,0], train_x[:,1], color=c)
plt.show()

Create the TensorFlow network and run the gradient descent algorithm to plot the results.

In [ ]:
import tensorflow
from tensorflow.keras.models     import Sequential
from tensorflow.keras.layers     import Dense, Input
from tensorflow.keras.optimizers import SGD

tensorflow.keras.backend.clear_session()

n_hidden   = 8   # The number of hidden units

model = Sequential()
model.add(Input(shape=(n_features,)))
model.add(Dense(units=n_hidden, activation='relu'))
model.add(Dense(units=n_labels, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer=SGD(learning_rate=0.05), metrics=['accuracy'])

model.fit(train_x, train_y, epochs=5000, batch_size=m, verbose=0)

prediction = model.predict(train_x, batch_size=m)

c = [None]*m
threshold = 0.6

for i in range(m):
    ix = np.argmax(prediction[i,:])   # The color with the highest probability
    c[i] = ('r', 'g', 'b', 'm', 'c', 'y')[ix]
    if prediction[i,ix] < threshold:
        c[i] = 'k'

plt.figure(1, figsize=(15, 10))
plt.scatter(train_x[:,0], train_x[:,1], color=c)
plt.show()

Plot the decision boundary using a new set of test data distinct from the training data, taking the label that has the highest probability as the predicted output.

In [ ]:
# Random test data
test_m = 10000

# Test data in a square grid
p = 100
test_x = np.empty((test_m,n_features)).astype(np.float32)
k = 0
for xc in np.linspace(-spread,spread,p):
    for yc in np.linspace(-spread,spread,p):
        test_x[k] = (xc, yc)
        k += 1

prediction = model.predict(test_x, batch_size=test_m)

c = [None for i in range(test_m)]

# Color each test point according to the label with the highest probability
for i in range(test_m):
    ix = np.argmax(prediction[i,:])
    c[i] = ('r', 'g', 'b', 'm', 'c', 'y')[ix]
    if prediction[i,ix] < 0.8:
        c[i] = 'k'

plt.figure(1, figsize=(15, 10))
plt.scatter(test_x[:,0], test_x[:,1], color=c)
plt.show()

Now try increasing the number of colors (n_labels), the number of clusters per color (n_clusters), the distance between the clusters (spread), the number of datapoints (m), and re-run the code above. As the problem gets more complicated, you will need to increase the number of hidden units (n_hidden) and possibly increase the number of training steps (n_steps).